# Lab Instructions

In the lab, you're presented a task such as building a dataset, training a model, or writing a training loop, and we'll provide the code structured in such a way that you can fill in the blanks in the code using the knowledge you acquired in the chapters that precede the lab. You should be able to find appropriate snippets of code in the course content that work well in the lab with minor or no adjustments.

The blanks in the code are indicated by ellipsis (`...`) and comments (`# write your code here`).

In some cases, we'll provide you partial code to ensure the right variables are populated and any code that follows it runs accordingly.

```python
# write your code here
x = ...
```

The solution should be a single statement that replaces the ellipsis, such as:

```python
# write your code here
x = [0, 1, 2]
```

In some other cases, when there is no new variable being created, the blanks are shown like in the example below: 

```python
# write your code here
...
```

Although we're showing you only a single ellipsis (`...`), you may have to write more than one line of code to complete the step, such as:

```python
# write your code here
for i, xi in enumerate(x):
    x[i] = xi * 2
```

## Installation Notes

The dataset functions implemented in Chapter 12 can be easily imported from a set of helper functions we're making available for your convenience. You can download it from the following link:

`https://raw.githubusercontent.com/lftraining/LFD273-code/main/object_detection_functions.py`

In Google Colab, you can run the following command to download the file:

In [ ]:
!wget https://raw.githubusercontent.com/lftraining/LFD273-code/main/object_detection_functions.py

Once the file is downloaded, you only need to import the required functions:

In [ ]:
from object_detection_functions import *

## 12.8 Lab 5A: Overview

In this first part of Lab 5, you will prepare the Oxford-IIIT Pet dataset for object detection.

You will parse the XML annotations, build COCO-style metadata, create a Hugging Face Dataset, attach Torchvision v2 transformations, and create the DataLoaders that will be consumed by the detector in Lab 5B.

### 12.8.1 Oxford-IIIT Pet Dataset

You'll build a dataset using the images and annotations from the Oxford-IIIT Pet dataset:

> "_We have created a 37 category pet dataset with roughly 200 images for each class. The images have a large variations in scale, pose and lighting. All images have an associated ground truth annotation of breed, head ROI, and pixel level trimap segmentation._"

First, we download images and annotations for the dataset and extract them locally:


In [ ]:
!wget https://thor.robots.ox.ac.uk/~vgg/data/pets/annotations.tar.gz
!wget https://thor.robots.ox.ac.uk/~vgg/data/pets/images.tar.gz
# Alternative mirror
# !wget https://datasets.cms.waikato.ac.nz/ufdl/data/oxford-pets/annotations.tar.gz
# !wget https://datasets.cms.waikato.ac.nz/ufdl/data/oxford-pets/images.tar.gz

In [ ]:
import tarfile
# Open and extract the archive automatically detecting compression (mode='r:*')
with tarfile.open('annotations.tar.gz', 'r:*') as tar:
    tar.extractall()
print("Annotations extracted successfully!")
with tarfile.open('images.tar.gz', 'r:*') as tar:
    tar.extractall()
print("Images extracted successfully!")

### 12.8.2 Annotations

The annotations follow the Pascal VOC challenge format, and are stored as individual XML files, one for each annotated image, inside the oxford-iiit-pet/annotations/xmls subfolder. Use the `xml_to_csv()` helper function to convert all these files into a Pandas dataframe and inspect its contents.

In [ ]:
# write your code here
xml_df = ...
xml_df.head()

The annotations contain the box coordinates in the Pascal VOC system (`[xmin, ymin, xmax, ymax]`), but they only have two main classes, cats and dogs, instead of the expected 37 classes found in the description. As it turns out, there are more files in the annotations folder, namely, `list.txt`, `trainval.txt`, and `test.txt`.

If you're in Google Colab, the command below will list the files inside the annotations folder:

In [ ]:
# if you chose a different root folder, change it accordingly
!ls -l ./annotations

Let's take a look at the list.txt file.

If you're in Google Colab, the command below will show you the first few lines of the list.txt file:

In [ ]:
!head ./annotations/list.txt

It contains a list of all images in the dataset, organized in four columns separated by spaces: Image, CLASS-ID, SPECIES, BREED ID. As it turns out, the "class" from the XML file is actually the species. We're interested in the true class ids, from 1 to 37, as stated in the description.

Now, let's take a look at the file corresponding to the data you loaded, the trainval split. If you're in Google Colab, the command below will show you the first few lines of the trainval.txt file:

In [ ]:
!head ./annotations/trainval.txt

It clearly follows the same structure as the previous file, but it does not contain any headers, and it lists only the images that belong to the original train and validation split.

We can load it in Pandas for easier visualization (just run the code below as is to visualize the dataframe with the information from the trainval.txt file):

In [ ]:
import pandas as pd

trainval_df = pd.read_csv('annotations/trainval.txt', sep=' ', header=None, names=['filename', 'class_id', 'species', 'breed_id'])
trainval_df

Each filename has its own corresponding class index (class_id), but the label itself, as the descriptive name corresponding to the category, is only available as part of the filename itself. 

We can easily extract it, though. Just run the code below as is to create a new column (category) in the dataframe:


In [ ]:
trainval_df['category'] = trainval_df['filename'].apply(lambda v: ' '.join([w.capitalize()
                                                                            for w in v.split('_')[:-1]]))

Moreover, there are 3,680 rows, one for each image, but there are 3,687 annotations retrieved from the XML files. Why? It is important to highlight that:
- some images may have more than one annotation/box - you saw that already in the Penn-Fudan dataset
- some images probably have no annotations/boxes (you'll see that soon)

We'll use the same custom dataset class ObjDetectionDataset once again, since it is prepared to take a CSV file or Pandas dataframe containing the annotations (filename, labels, xmin, ymin, xmax, and ymax columns), but keep in mind that only the filenames in the file/dataframe are going to be considered by it.

Therefore, we need to build an annotations file/dataframe that includes filenames that have no annotations as well. It is better to keep images without annotations as negative cases, so we merge both dataframes and make sure that:
- every filename is kept, so there are still 3,680 unique filenames after merging
- the resulting dataframe has, at least, the following columns: filename, label, category, xmin, ymin, xmax, and ymax

Run the code below as is to build the corresponding dataframe of annotations:

In [ ]:
trainval_df['filename'] = trainval_df['filename'].apply(lambda v: f'{v}.jpg')
annotations_df = trainval_df.merge(xml_df, how='left', on='filename')

colnames = ['filename', 'label', 'category', 'width', 'height', 'xmin', 'ymin', 'xmax', 'ymax']
annotations_df = annotations_df.rename(columns={'class_id': 'label'})[colnames]
annotations_df

Besides, we'll use the resulting dataframe to build a id2label dictionary to map class id into the corresponding category. Run the code below as is to build the dictionary:

In [ ]:
id2label = dict(annotations_df[['label', 'category']].drop_duplicates().values)
id2label

Let's run some assert commands to ensure everything is as expected. Run the code below as is. It shouldn't raise any errors nor produce any output. If an error is raised, you should double-check the code for loading the dataset and its annotations.

In [ ]:
assert len(annotations_df['filename'].unique()) == 3680
assert len(id2label.values()) == 37
assert len(annotations_df) == 3681

Shouldn'it be 3,687? Perhaps even more, since it should also include images without any annotations? It actually should, but some of the annotated images were excluded from the trainval.txt list of files for some unknown reason. In case you're curious, these are the images. 

Run the code below as is to visualize the extra annotations:

In [ ]:
extra_annotations = set(xml_df['filename'].unique()).difference(set(annotations_df['filename'].unique()))
extra_annotations

The whole point of this apparent detour from our main job here - fine-tuning an object detection model - is to illustrate the fact that every dataset has its issues, and you should always take your time to investigate how it's organized, if there are quality issues, and ensure it's in the right shape to be loaded into an instance of your dataset class.

So, let's get rid of any annotation that has something missing:

In [ ]:
annotations_df = annotations_df.dropna().copy()

### 12.8.3 Building the Dataset

The XML files use opposite corners. We’ll compute width and height so each box can be stored as `[x, y, width, height]`. We’ll also convert the original labels from 1..37 to the zero-based 0..36 convention used in the Hugging Face dataset.

In [ ]:
# write your code here
annotations_df["box_width"] = ...
annotations_df["box_height"] = ...

# make classes zero-based for our HF-style dataset
# write your code here
annotations_df["label"] = ...

Next, we’ll create one metadata record per image, keeping bounding boxes and categories as lists so the representation also works for images with multiple objects.

In [ ]:
import json

records = []
for filename, group in annotations_df.groupby("filename", sort=False):
    objects = {
        "bbox": [[float(row.xmin), float(row.ymin), float(row.box_width), float(row.box_height)]
                for row in group.itertuples()],
        "category": [int(label) for label in group["label"]],
    }
    record = {
        "file_name": filename,
        "width": int(group.iloc[0]["width"]),
        "height": int(group.iloc[0]["height"]),
        "objects": objects,
        "label": int(group.iloc[0]["label"])
    }
    records.append(record)

records[0]

Then, we save the metadata to a file named metadata.jsonl, and use it as an argument of our `build_object_detection_dataset()` helper function (developed in Chapter 12).

In [ ]:
from pathlib import Path

metadata_file = Path("metadata.jsonl")
with metadata_file.open("w", encoding="utf-8") as f:
    f.writelines([json.dumps(record) + '\n' for record in records])

In [ ]:
image_folder = './images'
# write your code here
pets_detection = ...

In order to split the dataset in a stratified fashion, so every class is represented in every split, we must cast the image-level breed label to `ClassLabel` first:

In [ ]:
from datasets import ClassLabel
pets_detection = pets_detection.cast_column("label", ClassLabel(num_classes=37))

Let’s check one example from our dataset now:

In [ ]:
pets_detection[0]

### 12.8.4 Train-Validation Splits

The original list of files does not give any indication regarding the split between training and validation sets, so you'll have to do it yourself.

Our suggestion is to shuffle the filenames, and take a large part of them (e.g. 80%) as training set, and the remaining files as validation set. Do not forget to stratify it by label (using the stratify_by_column argument):

In [ ]:
# write your code here
datasets = ...
datasets["validation"] = datasets.pop("test")
datasets

### 12.8.5 Data Augmentation

It is time to write your own training and validation transformation functions. First, define the composed transformations for each split:

- if it is the validation set, it should stick to the basics
    - sanitize bounding boxes, just in case
    - convert, if needed, PIL images to tensors
    - normalize the values
- if it is in the training set, it may perform data augmentation as well:
    - choose one or more data augmenting transformations
    - sanitize bounding boxes, just in case
    - convert, if needed, PIL images to tensors

Then, use them as arguments to  our own `build_detection_transform()` developed in Chapter 12, which builds the output dictionary expected by an object detection model.

In [ ]:
from torchvision.transforms import v2

# write your code here
train_transforms = ...
val_transforms = ...

# write your code here
train_fn = ...
val_fn = ...

### 12.8.6 Datasets and DataLoaders

The transformation functions can be lazily attached to each corresponding split using the datasets `with_transform()` method:

In [ ]:
# write your code here
datasets['train'] = ...
datasets['validation'] = ...

len(datasets['train']), len(datasets['validation'])

Next, create two data loaders, one for each dataset. You should shuffle the training set, but not the validation one. Also, keep batch size small (e.g. four) to avoid out-of-memory issues in the GPU.

In [ ]:
from torch.utils.data import DataLoader

dataloaders = {}
# write your code here
dataloaders['train'] = ...
dataloaders['validation'] = ...

Try fetching a mini-batch from your training set. Just run the code below as is:

In [ ]:
next(iter(dataloaders['train']))

Did you get an error? No? Consider yourself lucky! At some point, it will raise an error, whenever it tries to stack images with different sizes.

The collate function is the function used by the data loader to patch together multiple data points into a mini-batch. If your dataset is nothing but tensors, that's trivial: it only has to stack them up. Stacking them up, though, assumes every data point has exactly the same shape for its features.

In object detection models, though, this is not guaranteed to be the case: not only images may have different sizes, but also one image may have no boxes, another one may have three boxes, and yet another one may have only one. Those cannot be stacked together.
The solution, fortunately, is pretty easy, and it looks like this:

In [ ]:
def collate_fn(batch):
    images = [item["image"] for item in batch]
    targets = [item["target"] for item in batch]

    return images, targets

Throw the function above as the `collate_fn` argument of your data loaders, and try again:

In [ ]:
dataloaders = {}
# write your code here
dataloaders['train'] = ...
dataloaders['validation'] = ...

In [ ]:
next(iter(dataloaders['train']))